In [0]:
# Import necessary data types from pyspark.sql.types to define the schema
from pyspark.sql.types import StructType,StructField, StringType, IntegerType,DateType

# Define the schema for the sales CSV file
# Each StructField specifies the column name, data type, and whether it is nullable (True)
schema=StructType([ 
    StructField("Product_Id",IntegerType(),True),      # Unique identifier for each product
    StructField("Customer_Id",StringType(),True),      # Unique identifier for each customer
    StructField("Order_Date",DateType(),True),         # Date on which the order was placed
    StructField("Location",StringType(),True),         # Location where the order was made (e.g., country)
    StructField("Source_Order",StringType(),True),      # Source of the order (e.g., Restaurant, Online)
    ])

# Read the sales CSV file using the explicitly defined schema
# inferSchema is set to true as a fallback, but the explicit schema takes precedence
sales_df = spark.read.format("csv").option("inferSchema", "true").schema(schema).load("/Volumes/pyspark_project/spark/datafiles/SalesData/sales.csv.txt")

# Display the contents of the sales DataFrame
display(sales_df)


### ## **Adding Year Month Quarter it is used to show the quarter

In [0]:
# Import date-related functions from pyspark.sql.functions
# These functions extract the month, year, and quarter from a date column
from pyspark.sql.functions import month, year, quarter

# Extract the month (1-12) from the Order_Date column and store it as a new column "order_month"
sales_df = sales_df.withColumn("order_month", month("Order_Date"))

# Extract the year (e.g., 2023) from the Order_Date column and store it as a new column "order_year"
sales_df = sales_df.withColumn("order_year", year("Order_Date"))

# Extract the quarter (1-4) from the Order_Date column and store it as a new column "order_quarter"
sales_df = sales_df.withColumn("order_quarter", quarter("Order_Date"))

# Display the updated DataFrame with the new month, year, and quarter columns
display(sales_df)

In [0]:
# Import necessary data types from pyspark.sql.types to define the schema
from pyspark.sql.types import StructType,StructField, StringType, IntegerType,DataType

# Define the schema for the menu CSV file
# Each StructField specifies the column name, data type, and whether it is nullable (True)
schema=StructType([ 
    StructField("Product_Id",IntegerType(),True),      # Unique identifier for each product
    StructField("Product_Name",StringType(),True),     # Name of the product
    StructField("Price",StringType(),True)             # Price of the product
 ])

# Read the menu CSV file using the explicitly defined schema
# inferSchema is set to true as a fallback, but the explicit schema takes precedence
menu_df = spark.read.format("csv").option("inferSchema", "true").schema(schema).load("/Volumes/pyspark_project/spark/datafiles/ManuData/menu.csv.txt")

# Display the contents of the menu DataFrame
display(menu_df)


### ## **KPI For Business**

In [0]:
# Import the sum aggregation function from pyspark.sql.functions
from pyspark.sql.functions import sum

# Join the sales DataFrame with the menu DataFrame on Product_Id to get product prices
# Group the joined data by Customer_Id and calculate the total amount spent by each customer
# The sum of Price is aliased as "TotalSpent" for clarity
# Finally, order the results by customer_id in ascending order
customer_spent=sales_df.join(menu_df, sales_df.Product_Id == menu_df.Product_Id).groupBy("Customer_Id").agg(sum("Price").alias("TotalSpent")).orderBy('customer_id')

# Display the total amount spent by each customer
display(customer_spent)


Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import sum,col
Food_spent=menu_df.join(sales_df, sales_df.Product_Id == menu_df.Product_Id).groupBy("Product_name").agg(sum("Price").alias("Sum OF Price")).orderBy('Product_name')
display(Food_spent)


Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import sum,col
EachMont_spent=menu_df.join(sales_df, sales_df.Product_Id == menu_df.Product_Id).groupBy("order_month").agg(sum("Price").alias("Sum OF Price")).orderBy('order_month')
display(EachMont_spent)


Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import sum,col
EachYear_spent=menu_df.join(sales_df, sales_df.Product_Id == menu_df.Product_Id).groupBy("order_year").agg(sum("Price").alias("Sum OF Price")).orderBy('order_year')
display(EachYear_spent)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import sum,col
EachQuerter_spent=menu_df.join(sales_df, sales_df.Product_Id == menu_df.Product_Id).groupBy("order_quarter").agg(sum("Price").alias("Sum OF Price")).orderBy('order_quarter')
display(EachQuerter_spent)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import count,col
Total_order_purchased=menu_df.join(sales_df, sales_df.Product_Id == menu_df.Product_Id).groupBy(menu_df["Product_Id"],"Product_Name").agg(count(menu_df["Product_Id"]).alias("ProductCount")).orderBy('Product_Id','ProductCount')
display(Total_order_purchased)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import count,col
Total_order_purchased=menu_df.join(sales_df, sales_df.Product_Id == menu_df.Product_Id).groupBy(menu_df["Product_Id"],"Product_Name").agg(count(menu_df["Product_Id"]).alias("ProductCount")).orderBy('ProductCount',ascending=0).limit(3)
display(Total_order_purchased)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import count,col
Top_order=menu_df.join(sales_df, sales_df.Product_Id == menu_df.Product_Id).groupBy(menu_df["Product_Id"],"Product_Name").agg(count(menu_df["Product_Id"]).alias("ProductCount")).orderBy('ProductCount',ascending=0).limit(1)
display(Top_order)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import count,col,countDistinct
Customer_visted=sales_df.filter(sales_df.Source_Order=='Restaurant').groupBy("Customer_Id").agg(countDistinct("Order_Date").alias("CustomerVisited")).orderBy('Customer_Id','CustomerVisited',ascending=1)
display(Customer_visted)

In [0]:
from pyspark.sql.functions import count,col,countDistinct,sum
EachQuerter_spent=menu_df.join(sales_df, sales_df.Product_Id == menu_df.Product_Id).groupBy("location").agg(sum("Price").alias("TotalSalesByCountry")).orderBy('TotalSalesByCountry')
display(EachQuerter_spent)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import count,col,countDistinct,sum
EachQuerter_spent=menu_df.join(sales_df, sales_df.Product_Id == menu_df.Product_Id).groupBy("Source_Order").agg(sum("Price").alias("TotalSalesByCountry")).orderBy('TotalSalesByCountry')
display(EachQuerter_spent)

Databricks visualization. Run in Databricks to view.